# 01 — Synthetic Dataset Generation
## Personalised Bayesian Knowledge Tracing for Children with Mild Autism Spectrum Disorder (Ages 3–10)

---

**Author:** WASSWA COSMAS MUYOMBA  
**Date:** March 2026  
**Project:** Modified Bayesian Knowledge Tracing (BKT) Mobile Learning App — ASD Edition  
**Notebook role:** `01 / data_generation` → feeds `02 / validation` → `03 / bkt_fitting` → `04 / personalisation`

---

### 1. Objective

Generate a high-fidelity **synthetic longitudinal interaction dataset** that faithfully mirrors documented math-learning patterns in children with mild Autism Spectrum Disorder (ASD). Because real patient data for this age group is scarce, ethically complex to collect, and rarely longitudinal, simulation is the standard starting point for ASD adaptive-learning research (Dharsika et al., 2026; Polo-Blanco et al., 2022).

The simulator extends the classical four-parameter BKT model (Corbett & Anderson, 1994) with three ASD-specific additions:

| Extension | Motivation | Key reference |
|-----------|-----------|---------------|
| **Elevated forgetting** (`p_f`) | ASD learners show faster skill decay between sessions | Lee et al., 2023; Khajah et al., 2016 |
| **Variable engagement** (`behavior_score`) | Attention regulation deficits cause within-session performance swings | Jaime & Lewine, 2022 |
| **Higher hint reliance** | ASD children request scaffolding significantly more often than TD peers | Polo-Blanco et al., 2022 |
| **BKT posterior tracking** (`p_l_hat`) | Records the model's running belief about mastery — the core signal for personalisation | Corbett & Anderson, 1994 |

---

### 2. Skills Modelled (App Modules)

Six math skills are ordered by approximate cognitive demand for ages 3–10:

```
Easy  ──▶  counting  |  shape_recognition  |  addition
Hard  ──▶  subtraction  |  multiplication  |  division
```

Hard skills receive a stronger `difficulty_factor = 0.58` that dampens the transition probability, making the learning curve easier for BKT to identify while still preserving the easy-vs-hard gap reported by Dharsika et al. (2026).

---

### 3. BKT Parameter Calibration

All priors are sampled **per student** to preserve realistic between-child variability. Ranges are grounded in the cited literature:

| Parameter | ASD range | Typical range | Source |
|-----------|-----------|---------------|--------|
| `p_l0` — initial knowledge | N(0.45, 0.18) clipped [0.05, 0.95] | N(0.55, 0.12) | Tonizzi et al., 2023 |
| `p_t` — learning transition | U(0.14, 0.30) | U(0.18, 0.38) | Dharsika et al., 2026 |
| `p_f` — forgetting | U(0.05, 0.14) | U(0.01, 0.05) | Lee et al., 2023 |
| `p_g` — guessing | U(0.12, 0.24) | U(0.08, 0.20) | Dharsika et al., 2026 |
| `p_s` — slipping | U(0.12, 0.24) | U(0.08, 0.20) | Bejarano-Martín et al., 2024 |
| `behavior_mean` | U(0.62, 0.82) | U(0.78, 0.92) | Jaime & Lewine, 2022 |

---

### 4. Output Schema

| Column | Type | Description |
|--------|------|-------------|
| `anon_student_id` | int | Anonymised learner index |
| `skill_name` | str | One of the six math skills |
| `opportunity` | int | Practice attempt number (1 → 120) |
| `correct` | 0/1 | Observed response correctness |
| `hint_used` | 0/1 | Whether scaffolding hint was requested |
| `behavior_score` | float [0,1] | Momentary engagement proxy |
| `p_l_hat` | float [0,1] | BKT posterior: P̂(mastered \| evidence so far) |

---

### 5. References

- Bejarano-Martín, Á. et al. (2024). *Technology-based interventions in ASD: Systematic review.* J. Autism Dev. Disord.
- Corbett, A. T., & Anderson, J. R. (1994). *Knowledge tracing: Modelling the acquisition of procedural knowledge.* User Modelling and User-Adapted Interaction, 4, 253–278.
- Dharsika, B. et al. (2026). *BKT-based adaptive daily living skills for ASD.* Computers & Education.
- Jaime, M., & Lewine, J. D. (2022). *Attention variability in ASD during learning tasks.* Frontiers in Neuroscience.
- Khajah, M. M. et al. (2016). *How deep is knowledge tracing?* EDM Conference.
- Lee, J. et al. (2023). *Forgetting-augmented BKT for long-term modelling.* LAK Conference.
- Polo-Blanco, I. et al. (2022). *Teaching mathematics to students with ASD.* Mathematics, 10(3), 341.
- Tonizzi, I. et al. (2023). *Arithmetic difficulties in ASD: Meta-analysis.* Research in Developmental Disabilities.

---

In [ ]:
# ================================================
# Imports & Setup
# ================================================

# Import numpy for numerical operations and random sampling
import numpy as np

# Import pandas for tabular data handling (creating our final dataset)
import pandas as pd

# Import Path from pathlib for cross-platform file path management
from pathlib import Path

# Fix the Random Number Generator (RNG) seed.
# This ensures that every time we run this generation script, we get the exact same dataset,
# which is essential for reproducible experiments and consistent downstream model evaluations.
np.random.seed(42)

# Define the project path where the raw synthetic CSV files will be saved.
# We navigate up one directory ("../") and into "data/raw".
DATA_RAW = Path("../data/raw")

# Create the output directory if it doesn't already exist.
# parents=True allows creating intermediate directories, and exist_ok=True prevents errors if it already exists.
DATA_RAW.mkdir(parents=True, exist_ok=True)

# Print a quick checkpoint message to verify the notebook setup has successfully executed.
print("Setup complete")

Setup complete


In [ ]:
# ================================================
# Autism-Tuned Synthetic BKT Simulator
# ================================================
def generate_synthetic_bkt_data(
    n_students: int = 1200,
    n_opportunities_per_skill: int = 120,
    skills: list = None,
    autism_mode: bool = True,
    output_path: str | Path = None,
    seed: int = 42,
    session_length: int = 10,
) -> pd.DataFrame:
    """
    Simulator tuned for mild autism (ages 3-10).

    Key modelling choices:
    - learning happens at the attempt level;
    - forgetting is applied mainly at session boundaries instead of every single item;
    - p_l_hat is tracked explicitly for downstream validation notebooks.

    This keeps the ASD profile slower and more fragile than the typical profile,
    but still learnable enough for BKT to identify non-degenerate priors.
    """
    # Use a default list of mathematical/cognitive skills commonly taught in early childhood
    if skills is None:
        skills = [
            "counting",
            "shape_recognition",
            "addition",
            "subtraction",
            "multiplication",
            "division",
        ]

    # Initialize a random number generator specifically for this function to maintain deterministic outputs
    rng = np.random.default_rng(seed)
    
    # We will accumulate all generated learner trajectories (interactions) here
    data = []

    # Simple skills generally have higher baseline likelihoods for learning
    easy_skills = {"counting", "shape_recognition", "addition"}

    for student_id in range(n_students):
        # 1. Define base cognitive traits for the student
        # Prior knowledge (p_l0): Lower and more variable for ASD profiles due to scattered skill mastery
        p_l0 = np.clip(rng.normal(0.42 if autism_mode else 0.55, 0.16 if autism_mode else 0.12), 0.05, 0.95)
        
        # Base transition/learning rate (p_t): Usually lower for ASD learners acquiring novel skills
        base_p_t = rng.uniform(0.14, 0.30) if autism_mode else rng.uniform(0.18, 0.38)
        
        # Forgetting probability (p_f): Autism-tuned profiles frequently have higher regression bridging sessions
        p_f = rng.uniform(0.05, 0.14) if autism_mode else rng.uniform(0.01, 0.04)
        
        # Guess (p_g) and Slip (p_s): Higher general noise bounds for ASD populations due to attentional shifts
        p_g = rng.uniform(0.12, 0.24) if autism_mode else rng.uniform(0.08, 0.20)
        p_s = rng.uniform(0.12, 0.24) if autism_mode else rng.uniform(0.08, 0.20)
        
        # Behavior trait proxy map: how engaged is the student on average?
        behavior_mean = rng.uniform(0.62, 0.82) if autism_mode else rng.uniform(0.78, 0.92)

        for skill in skills:
            # 2. Skill-specific modifications
            # Difficulty scales down the learning rate. 'Hard' skills get a heavier penalty in ASD mode
            difficulty_factor = 1.0 if skill in easy_skills else (0.58 if autism_mode else 0.82)
            p_t = base_p_t * difficulty_factor

            # Assign true underlying start knowledge (1=mastered, 0=not mastered)
            knowledge = int(rng.random() < p_l0)
            
            # The simulator's "own" internal estimate/tracking of the learner's knowledge probability
            p_l_hat = float(p_l0)

            for opp in range(n_opportunities_per_skill):
                # Sample engagement specific to this opportunity. ASD engagement is generally more volatile
                behavior = float(np.clip(rng.normal(behavior_mean, 0.20 if autism_mode else 0.14), 0.0, 1.0))

                # Assess hint usage probability:
                # - Driven up by low engagement OR low internal estimation of mastery (p_l_hat).
                low_engagement = behavior < (0.45 if autism_mode else 0.35)
                if autism_mode:
                    hint_prob = 0.18 + 0.22 * int(low_engagement) + 0.22 * int(p_l_hat < 0.55)
                else:
                    hint_prob = 0.08 + 0.10 * int(low_engagement) + 0.10 * int(p_l_hat < 0.60)
                hint_prob = float(np.clip(hint_prob, 0.05, 0.75))
                hint_used = int(rng.random() < hint_prob)

                # 3. Dynamic BKT parameter adjustments per interaction
                # Engagement and hints dynamically increase or decrease learning likelihood
                learning_readiness = (0.70 + 0.30 * behavior) if autism_mode else (0.82 + 0.18 * behavior)
                effective_p_t = float(np.clip(p_t * learning_readiness * (1 + 0.12 * hint_used), 0.02, 0.55))
                
                # Slips happen more when behavior is poor; Guessing decreases when behavior is good but rises with hints
                effective_p_s = float(np.clip(p_s * (1.05 - 0.30 * behavior), 0.02, 0.28))
                effective_p_g = float(np.clip(p_g * (1.00 - 0.10 * behavior) + 0.04 * hint_used, 0.04, 0.30))

                # 4. Generate the observable event (the correctness of the learner's answer)
                if knowledge == 1:
                    # If they know it, they answer right unless they slip
                    correct = 1 - rng.binomial(1, effective_p_s)
                else:
                    # If they don't know it, they answer wrong unless they guess correctly
                    correct = rng.binomial(1, effective_p_g)

                # 5. Bayesian Posterior update of knowledge belief tracking (Standard BKT formula)
                if correct == 1:
                    denom = p_l_hat * (1 - effective_p_s) + (1 - p_l_hat) * effective_p_g
                    posterior = p_l_hat if denom <= 0 else (p_l_hat * (1 - effective_p_s)) / denom
                else:
                    denom = p_l_hat * effective_p_s + (1 - p_l_hat) * (1 - effective_p_g)
                    posterior = p_l_hat if denom <= 0 else (p_l_hat * effective_p_s) / denom
                posterior = float(np.clip(posterior, 0.001, 0.999))

                # 6. Apply Forgetting (A core ASD modelling step)
                # Forgetting is concentrated at the end of a 'session' (e.g., between days or class periods)
                session_boundary = ((opp + 1) % session_length == 0)
                if session_boundary:
                    effective_p_f = float(np.clip(p_f * (1.05 - 0.25 * behavior), 0.0, 0.18))
                elif autism_mode and behavior < 0.20:
                    # ASD mode uniquely applies minor forgetting during extremely poor behavioral episodes
                    effective_p_f = float(np.clip(0.12 * p_f, 0.0, 0.04))
                else:
                    effective_p_f = 0.0

                # Log the generated interaction into our trajectory table
                data.append({
                    "anon_student_id": student_id,
                    "skill_name": skill,
                    "correct": int(correct),
                    "hint_used": hint_used,
                    "behavior_score": round(behavior, 3),
                    "opportunity": opp + 1,
                    "p_l_hat": round(posterior, 4),
                })

                # 7. Roll forward true latent knowledge states
                if knowledge == 1:
                    # Check for regression/forgetting
                    knowledge = 0 if rng.random() < effective_p_f else 1
                else:
                    # Check for skill mastery/learning transition
                    if rng.random() < effective_p_t:
                        knowledge = 1

                # Update the simulated tracker for the next step, incorporating both slipping/guessing likelihood and forgetting.
                p_l_hat = float(np.clip(
                    posterior * (1 - effective_p_f) + (1 - posterior) * effective_p_t,
                    0.001,
                    0.999,
                ))

    # Collect list of dictionaries into a Pandas structured frame
    df = pd.DataFrame(data)
    
    # Save directly to disk if requested
    if output_path:
        df.to_csv(output_path, index=False)
        print(f"Saved {len(df):,} rows to {output_path}")
    return df

# ================================================
# Generate BOTH datasets
# ================================================
# Build the Autism dataset utilizing the ASD parameters
df_autism = generate_synthetic_bkt_data(
    n_students=1200,
    autism_mode=True,  # Turns on the ASD profiling overrides
    output_path=DATA_RAW / "synthetic_autism_data.csv",
    seed=42,
    session_length=10,
)

# Build the Typical (Neurotypical) baseline comparison dataset
df_typical = generate_synthetic_bkt_data(
    n_students=800,
    autism_mode=False, # Standard BKT profile
    output_path=DATA_RAW / "synthetic_typical_data.csv",
    seed=43,
    session_length=10,
)

# Validation print indicating shapes correctly match expected rows
print("Generation complete. Autism dataset shape:", df_autism.shape)

Saved 864,000 rows to ..\data\raw\synthetic_autism_data.csv
Saved 576,000 rows to ..\data\raw\synthetic_typical_data.csv
Generation complete. Autism dataset shape: (864000, 6)


### Summary Statistics (Autism vs Typical)
Run the cell below to preview — these will be used in our Notebook 02 for statistical validation.

In [ ]:
# ================================================
# Preview and Summarize Generated Datasets
# ================================================

# Print a heading for quick visual separation in the notebook console output
print("Autism dataset sample:")

# Group the dataframe by 'skill_name' to compute aggregated statistics 
# across all students and opportunities for the generated autism data.
# We focus on the mean of three core metrics:
# 1. correct: Average correct response rate for the skill
# 2. hint_used: Average percentage of hints requested across interactions 
# 3. behavior_score: The average simulated behavioral/engagement score 

display(df_autism.groupby("skill_name")[["correct", "hint_used", "behavior_score"]].mean().round(3))

Autism dataset sample:


,correct,hint_used,behavior_score
skill_name,,,
addition,0.517,0.326,0.657
counting,0.513,0.328,0.657
division,0.463,0.352,0.658
multiplication,0.459,0.350,0.658
shape_recognition,0.517,0.324,0.658
subtraction,0.463,0.350,0.657
